In [ ]:
!pip install mlflow xgboost scikit-learn pandas numpy

In [ ]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path
import joblib

In [ ]:
# load the model files from our project
MODEL_DIR = Path("../ai/model")

existing_model = joblib.load(MODEL_DIR / "xgboost_rul_model.pkl")
scaler = joblib.load(MODEL_DIR / "standard_scaler.pkl")
feature_names = joblib.load(MODEL_DIR / "feature_names.pkl")

print(type(existing_model))
print(len(feature_names), "features")

In [ ]:
# generate some dummy data to test with
np.random.seed(42)
n_samples = 1000
n_features = len(feature_names)

X = np.random.randn(n_samples, n_features)
w = np.random.randn(n_features) * 0.5
y = X @ w + np.random.randn(n_samples) * 10 + 125
y = np.clip(y, 0, 300)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train:", X_train.shape, "test:", X_test.shape)

In [ ]:
# first training run with mlflow
mlflow.set_experiment("Predictive Maintenance - Demo")

with mlflow.start_run(run_name="run1-default"):
    params = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "objective": "reg:squarederror"
    }
    mlflow.log_params(params)

    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")

    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

In [ ]:
# try different parameters
with mlflow.start_run(run_name="run2-more-trees"):
    params = {"n_estimators": 200, "max_depth": 8, "learning_rate": 0.1, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

with mlflow.start_run(run_name="run3-low-lr"):
    params = {"n_estimators": 150, "max_depth": 6, "learning_rate": 0.05, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

with mlflow.start_run(run_name="run4-shallow"):
    params = {"n_estimators": 80, "max_depth": 4, "learning_rate": 0.15, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

print("done")

In [ ]:
# log some prediction stats too
with mlflow.start_run(run_name="batch-prediction"):
    y_pred = model.predict(X_test)

    mlflow.log_metric("avg_rul", float(np.mean(y_pred)))
    mlflow.log_metric("min_rul", float(np.min(y_pred)))
    mlflow.log_metric("max_rul", float(np.max(y_pred)))
    mlflow.log_metric("std_rul", float(np.std(y_pred)))

    critical = int(np.sum(y_pred < 30))
    warning = int(np.sum((y_pred >= 30) & (y_pred < 75)))
    healthy = int(np.sum(y_pred >= 75))

    mlflow.log_metric("engines_critical", critical)
    mlflow.log_metric("engines_warning", warning)
    mlflow.log_metric("engines_healthy", healthy)

    print(f"avg RUL: {np.mean(y_pred):.1f}")
    print(f"critical: {critical}, warning: {warning}, healthy: {healthy}")

In [ ]:
# list all runs
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("Predictive Maintenance - Demo")
runs = client.search_runs(experiment.experiment_id)

for run in runs:
    d = run.data
    rmse_val = d.metrics.get('rmse')
    rmse_str = f"{rmse_val:.2f}" if rmse_val is not None else "N/A"
    print(run.info.run_name, "-> RMSE:", rmse_str)

In [ ]:
# to see the dashboard run this in terminal:
# mlflow ui
# then go to http://localhost:5000